# Code mẫu — Attention và Transformer

Ở bài embedding, ta gộp các token bằng một phép trung bình cố định. Attention thay phép trung bình ấy bằng một phép trộn có trọng số, và trọng số thay đổi theo query. Ta đi từ Q, K, V tính bằng tay, sang mask đặt đúng trục, rồi tới một lớp self-attention có che tương lai. Suốt trang, ta kiểm cả shape lẫn ý nghĩa, vì shape đúng chưa chắc đã đúng nghĩa.

## Ví dụ 1: Tính scaled dot-product attention bằng tay

Một query nhìn vào hai key. Score giữa query và từng key quyết định trọng số. Value là nội dung sẽ được trộn lại theo trọng số ấy để tạo đầu ra. Trước khi lấy `exp`, ta trừ đi giá trị lớn nhất của hàng để tránh tràn số. Việc này không làm đổi kết quả softmax.

In [ ]:
import numpy as np
import torch

torch.set_num_threads(1)
Q = np.array([[1.]], dtype=np.float64)
K = np.array([[1.], [0.]], dtype=np.float64)
V = np.array([[2., 0.], [0., 4.]], dtype=np.float64)
scores = Q @ K.T / np.sqrt(Q.shape[-1])
shifted = scores - scores.max(axis=-1, keepdims=True)
weights = np.exp(shifted)
weights /= weights.sum(axis=-1, keepdims=True)
output = weights @ V

torch_output = (
    torch.tensor(Q) @ torch.tensor(K).T
    / np.sqrt(Q.shape[-1])
).softmax(dim=-1) @ torch.tensor(V)
print("shapes Q/K/V:", Q.shape, K.shape, V.shape)
print("scores:", scores.round(6).tolist())
print("weights:", weights.round(6).tolist())
print("output:", output.round(6).tolist())
print("NumPy matches PyTorch:", np.allclose(output, torch_output.numpy()))
assert output.shape == (1, 2)
assert np.allclose(weights.sum(axis=-1), 1.)

shapes Q/K/V: (1, 1) (2, 1) (2, 2)
scores: [[1.0, 0.0]]
weights: [[0.731059, 0.268941]]
output: [[1.462117, 1.075766]]
NumPy matches PyTorch: True


Query `[1]` cho score 1 với key đầu và 0 với key sau. Softmax biến hai score thành trọng số khoảng 0.731 và 0.269. Đầu ra là $0.731\cdot[2,0]+0.269\cdot[0,4]$, tức khoảng `[1.462, 1.076]`. Đây là một phép trộn mềm của hai value, chứ không phải chọn hẳn value có score cao hơn.

Ghi nhớ shape tổng quát: `Q` là `(Lq, dk)`, `K` là `(Lk, dk)`, `V` là `(Lk, dv)`, và đầu ra là `(Lq, dv)`. Mỗi query cho ra một vector, có độ dài bằng độ dài của value.

## Ví dụ 2: Mask theo đúng trục query và key

Bảng score có shape `(B, Lq, Lk)`: hàng là query, cột là key. Có hai loại mask. Padding mask chặn cả một **cột**, tức một key là PAD, với mọi query. Causal mask chặn các ô mà query nhìn về key ở tương lai. Ta gộp hai mask lại rồi áp lên score trước khi softmax.

Lưu ý quy ước của PyTorch. Trong `nn.MultiheadAttention`, giá trị `True` trong mask boolean nghĩa là **cấm**. `key_padding_mask=True` cấm một key, còn `attn_mask=True` cấm một cặp query–key.

In [ ]:
import torch

torch.set_num_threads(1)
torch.manual_seed(392)
Q = torch.tensor([[[1., 0.], [0., 1.], [1., 1.], [0., 0.]]])
K = torch.tensor([[[1., 0.], [0., 1.], [1., 1.], [2., 2.]]])
V = torch.tensor([[[1., 0.], [0., 2.], [3., 3.], [9., 9.]]])
length = Q.shape[1]
padding_mask = torch.tensor([[False, False, False, True]])
causal_mask = torch.triu(torch.ones(length, length, dtype=torch.bool), diagonal=1)
scores = Q @ K.transpose(-2, -1) / (Q.shape[-1] ** 0.5)
blocked = padding_mask[:, None, :] | causal_mask[None, :, :]
masked_scores = scores.masked_fill(blocked, float("-inf"))
weights = masked_scores.softmax(dim=-1)
attended = weights @ V
future_weights = weights.masked_select(causal_mask.unsqueeze(0))
pad_weights = weights[:, :, -1]
print("score shape:", tuple(scores.shape))
print("blocked shape:", tuple(blocked.shape))
print("query 2 weights:", [round(float(v), 6) for v in weights[0, 2]])
print("max PAD-key weight:", round(float(pad_weights.abs().max()), 6))
print("max future-key weight:", round(float(future_weights.abs().max()), 6))
print("all row sums 1:", torch.allclose(
    weights.sum(dim=-1), torch.ones_like(weights[..., 0])))
print("PAD-query output norm:", round(float(attended[0, 3].norm()), 6))
assert weights.shape == (1, 4, 4)
assert float(pad_weights.abs().max()) == 0.
assert float(future_weights.abs().max()) == 0.
assert torch.allclose(weights.sum(dim=-1), torch.ones_like(weights[..., 0]))

score shape: (1, 4, 4)
blocked shape: (1, 4, 4)
query 2 weights: [0.248255, 0.248255, 0.50349, 0.0]
max PAD-key weight: 0.0
max future-key weight: 0.0
all row sums 1: True
PAD-query output norm: 2.134375


Padding mask có shape `(1, 4)`. Ta thêm một trục ở giữa để nó thành `(1, 1, 4)`, và broadcasting kéo nó ra mọi hàng query. Causal mask có shape `(4, 4)`, ta thêm trục batch ở đầu. Đặt sai trục thì code vẫn chạy, nhưng sẽ chặn nhầm hàng thay vì cột.

Query ở vị trí 2 chỉ được nhìn key 0, 1 và 2. Key 3 vừa là PAD vừa nằm ở tương lai, nên trọng số bằng 0. Mỗi hàng vẫn có tổng bằng 1, vì softmax chia lại trên những key còn được phép.

Dòng cuối dễ bị bỏ qua: hàng query 3, vốn là PAD, vẫn có đầu ra khác 0. Mask ở đây chỉ chặn PAD làm **nguồn** để nhìn vào, chứ không xóa đầu ra ở vị trí PAD. Khi gộp đầu ra hoặc tính loss, ta còn phải bỏ các vị trí PAD bằng một mask riêng, như đã làm ở bài embedding.

## Ví dụ 3: MultiheadAttention, và phép kiểm rò rỉ tương lai

Ta dùng lớp `nn.MultiheadAttention` có sẵn, với `batch_first=True` để giữ shape `(B, L, d)`. Dropout được tắt để kết quả ổn định. Cách kiểm causal mask rất trực tiếp: sửa mạnh token cuối, rồi xem ba đầu ra đầu có đổi không. Nếu mask đúng, ba vị trí đầu không thể nhìn thấy token cuối, nên đầu ra của chúng phải giữ nguyên.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(393)
block = nn.MultiheadAttention(
    embed_dim=4, num_heads=2, batch_first=True, dropout=0.
)
block.eval()
x = torch.tensor([[
    [1., 0., 0., 1.],
    [0., 1., 1., 0.],
    [1., 1., 0., 0.],
    [0., 0., 1., 1.],
]])
causal = torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)
changed = x.clone()
changed[:, 3] += torch.tensor([10., -3., 7., 2.])
with torch.no_grad():
    before, weights = block(
        x, x, x, attn_mask=causal,
        need_weights=True, average_attn_weights=False,
    )
    after, _ = block(
        changed, changed, changed, attn_mask=causal,
        need_weights=True, average_attn_weights=False,
    )
blocked_weights = weights[0].masked_select(causal.unsqueeze(0).expand_as(weights[0]))
print("output shape:", tuple(before.shape))
print("weights shape (B,H,Q,K):", tuple(weights.shape))
print("max future weight:", round(float(blocked_weights.abs().max()), 6))
print("first three outputs unchanged:", torch.allclose(
    before[:, :3], after[:, :3], atol=1e-6))
assert before.shape == (1, 4, 4)
assert weights.shape == (1, 2, 4, 4)
assert float(blocked_weights.abs().max()) == 0.
assert torch.allclose(before[:, :3], after[:, :3], atol=1e-6)

output shape: (1, 4, 4)
weights shape (B,H,Q,K): (1, 2, 4, 4)
max future weight: 0.0
first three outputs unchanged: True


Đầu ra giữ shape `(1, 4, 4)`, giống đầu vào. Ta đặt `average_attn_weights=False` để nhận trọng số riêng của từng head, nên trọng số có bốn trục `(batch, head, query, key)`. Khi vẽ hay in một ma trận attention, nhớ ghi rõ đó là của head nào.

Mọi trọng số nhìn về tương lai đều bằng 0, và sửa token cuối không làm đổi ba đầu ra đầu. Phép kiểm này mới xác nhận riêng một lớp attention. Một model Transformer hoàn chỉnh còn nhiều lớp khác và cả bước chuẩn bị dữ liệu, nên muốn chắc chắn không rò rỉ, phải kiểm theo cùng cách trên toàn bộ model.